# CipherMark — banc d'essai des modèles entraînés

**Mémoire de Master 2 · Université de Yaoundé I**
*Faculté des Sciences — Département d'Informatique*

Ce carnet-ci ne présente rien : il **fait tourner** les modèles. Vous chargez
une image, un modèle la tatoue ; vous reprenez l'image tatouée, un autre modèle
en extrait le champ témoin ; vous tapez une phrase, un générateur produit une
image déjà marquée.

---

## Avant de commencer — deux choses à régler

**1 · Activez le GPU.** Menu **Exécution → Modifier le type d'exécution →
Accélérateur matériel : GPU**. Les sections 1 à 4 tournent sans GPU, mais les
décodeurs génératifs (sections 5 à 7) exigent CUDA : le DC-AE utilise
`TritonRMSNorm`, qui n'a pas d'implémentation processeur.

**2 · Les modèles pèsent lourd.** Chaque section indique ce qu'elle télécharge
depuis votre Drive. Ne chargez que ce que vous voulez essayer.

| section | modèle | taille | GPU |
|---|---|---|---|
| 1–4 | tatoueur 64 / 96 / 128 bits, phase B | 462–514 Mo | non |
| 5 | décodeur génératif conditionné (phase D) | 5,8 Go | **oui** |
| 6 | injection latente (phase G) | 370 Mo | **oui** |
| 7 | prompt → image tatouée (phase E + SANA) | 2,4 Go + 9 Go | **oui** |

---
# 1 · Mise en route

Cette cellule monte votre Drive, récupère le code, installe les dépendances.
Comptez trois à cinq minutes la première fois.

In [ ]:
#@title Monter le Drive, installer, préparer  { display-mode: "form" }
import os, sys, subprocess, glob, textwrap, time

#@markdown Laissez vide pour que le carnet trouve le dossier tout seul.
RACINE_DRIVE = "" #@param {type:"string"}

t0 = time.time()
if not os.path.isdir("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

MARQUEUR = "00-code-source-a-jour.tar.gz"

def trouve_racine(indice=""):
    """Cherche le dossier ciphermark plutot que de supposer son chemin.

    Colab monte le Drive du compte connecte AU NAVIGATEUR, qui n'est pas
    forcement celui ou vivent les donnees. On cherche donc le dossier par son
    contenu -- l'archive de code -- sur les emplacements possibles : MyDrive,
    « My Drive » (ancien nom), les Drive partages, et « Partage avec moi ».
    """
    if indice.strip():
        c = indice.strip().rstrip("/")
        if os.path.isdir(c): return c, "chemin fourni"
        raise SystemExit(f"le chemin fourni n'existe pas : {c}")
    for motif in ("/content/drive/MyDrive/**/" + MARQUEUR,
                  "/content/drive/My Drive/**/" + MARQUEUR,
                  "/content/drive/Shareddrives/**/" + MARQUEUR,
                  "/content/drive/*/**/" + MARQUEUR):
        for f in glob.glob(motif, recursive=True):
            return os.path.dirname(f), "trouve par recherche"
    # rien : on montre ce qui EST monte, pour identifier le compte
    lignes = []
    for base in sorted(glob.glob("/content/drive/*")):
        lignes.append(f"    {base}/")
        for e in sorted(glob.glob(base + "/*"))[:14]:
            lignes.append(f"        {os.path.basename(e)}")
    raise SystemExit(textwrap.dedent(f"""
        Le dossier « ciphermark » n'a pas ete trouve.

        Ce qui est monte en ce moment :
        """) + "\n".join(lignes) + textwrap.dedent(f"""

        Deux causes possibles, et deux remedes.

        1. Colab est connecte a un AUTRE compte Google que celui qui contient
           les donnees. Le carnet a ete depose sur un Drive dont la racine
           contient « ciphermark », « Classroom » et « Colab ». Si la liste
           ci-dessus ne ressemble pas a cela, changez de compte : cliquez sur
           votre avatar en haut a droite de Colab, choisissez le bon compte,
           puis Execution -> Redemarrer la session.

        2. Le dossier est ailleurs, ou partage avec vous sans etre dans votre
           Drive. Reperez-le dans la liste ci-dessus et collez son chemin
           complet dans le champ RACINE_DRIVE, par exemple :
               /content/drive/MyDrive/travail/ciphermark
        """))

RACINE_DRIVE, comment = trouve_racine(RACINE_DRIVE)
print(f"  dossier de donnees : {RACINE_DRIVE}   ({comment})")

if not os.path.isdir("/content/code-memoire"):
    src = os.path.join(RACINE_DRIVE, MARQUEUR)
    subprocess.run(["tar", "xzf", src, "-C", "/content"], check=True)
    print("  code extrait")

os.chdir("/content/code-memoire")
sys.path[:0] = ["/content/code-memoire", "/content/code-memoire/deps"]

# Liste calee sur requirements.txt du depot. onnx, onnxsim et onnxruntime ne
# servent a aucun calcul ici, mais deps/efficientvit/apps/utils/export.py les
# importe au chargement du module : sans eux, l'import de DCAE_HF echoue.
paquets = ["omegaconf", "lpips", "torchmetrics", "timm", "einops", "reedsolo",
           "pycryptodome", "av", "opencv-python-headless", "safetensors",
           "pytorch_msssim", "onnx", "onnxsim", "onnxruntime",
           "pandas", "scipy", "scikit-image", "tensorboard"]
NOM_MODULE = {"opencv-python-headless": "cv2", "pycryptodome": "Crypto",
              "scikit-image": "skimage", "pytorch_msssim": "pytorch_msssim"}
manquants = []
for pq in paquets:
    try: __import__(NOM_MODULE.get(pq, pq.replace("-", "_")))
    except ImportError: manquants.append(pq)
if manquants:
    print("  installation de :", " ".join(manquants))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *manquants], check=True)
    # Un paquet fraichement installe n'est pas toujours visible d'un
    # interpreteur deja lance : on rafraichit le cache des chemins d'import.
    import importlib; importlib.invalidate_caches()
else:
    print("  toutes les dependances sont deja presentes")

import torch
GPU = torch.cuda.is_available()
# Le bfloat16 exige une carte Ampere ou plus recente (capacite de calcul >= 8.0).
# Colab en version gratuite fournit souvent un T4, architecture Turing, capacite
# 7.5 : les convolutions en bf16 y echouent avec « GET was unable to find an
# engine to execute this computation ». On choisit donc float16 dans ce cas.
if GPU:
    CAPACITE = torch.cuda.get_device_capability(0)
    DTYPE_AMP = torch.bfloat16 if CAPACITE[0] >= 8 else torch.float16
else:
    CAPACITE, DTYPE_AMP = (0, 0), torch.float32
print()
print(f"  torch {torch.__version__}   CUDA : {'OUI — ' + torch.cuda.get_device_name(0) if GPU else 'NON'}")
if GPU:
    print(f"  capacite de calcul {CAPACITE[0]}.{CAPACITE[1]} -> type flottant "
          f"{str(DTYPE_AMP).replace('torch.','')}"
          + ("  (bfloat16 indisponible sur cette carte)" if CAPACITE[0] < 8 else ""))
if not GPU:
    print("  ATTENTION : sans GPU, seules les sections 1 a 5 fonctionneront.")
print(f"  pret en {time.time()-t0:.0f} s")

# ═══════════════════════════════════════════════════════════════════════════
#  Chargeurs paresseux
#
#  Chaque cellule du carnet a besoin de modeles que d'autres cellules
#  chargent. Exiger le bon ordre d'execution est une mauvaise idee : on
#  tombe sur un NameError qui n'apprend rien. Les fonctions ci-dessous
#  chargent a la demande ce qui manque, et ne refont jamais un travail deja
#  fait.
# ═══════════════════════════════════════════════════════════════════════════
import shutil, warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("timm").setLevel(logging.ERROR)

CHEMINS_TATOUEUR = {
 "64 bits (reference)":           "01-tatoueur-modeles-de-reference-64-et-128-bits/phaseA2_64bits_stable_checkpoint.pth",
 "128 bits":                      "01-tatoueur-modeles-de-reference-64-et-128-bits/phaseA2_128bits_stable_checkpoint.pth",
 "96 bits":                       "02-tatoueur-96-bits-canal-plus-large/checkpoint.pth",
 "phase B (robuste aux attaques)":"03-tatoueur-robuste-aux-attaques-passives/checkpoint.pth",
}
SPEC_DECODEUR = {
 "phase E — decodeur de SANA (2,4 Go)":
   ("04b-decodeur-de-sana-conditionne-40000-pas-bit-acc-0.9867/checkpoint.pt",
    "dc-ae-f32c32-sana-1.0"),
 "phase D — autoencodeur ImageNet (5,8 Go)":
   ("04-decodeur-generatif-conditionne-sur-omega-modele-retenu/checkpoint.pt",
    "dc-ae-f64c128-in-1.0"),
}

def _copie_du_drive(rel):
    src = os.path.join(RACINE_DRIVE, rel)
    loc = "/content/ckpt/" + rel.replace("/", "__")
    os.makedirs("/content/ckpt", exist_ok=True)
    if not os.path.exists(loc):
        if not os.path.exists(src):
            raise SystemExit(f"absent du Drive : {src}")
        print(f"      copie de {os.path.getsize(src)/2**20:.0f} Mo depuis le Drive...")
        shutil.copy(src, loc)
    return loc

def charger_tatoueur(modele="64 bits (reference)", force=0.08, bavard=True):
    """Embedder + extracteur + hash + chaine CipherMark."""
    global WAM, CM, PHASE, PHASH, CLES, REGISTRE, NBITS, IMG_SIZE, DEVICE, TAU, HASH_BITS
    import torch
    from distseal.utils.cfg import get_config_from_checkpoint, setup_model_from_checkpoint
    from distseal.utils import optim as uoptim
    from distseal.ciphermark.phash import PerceptualHash, _try_load_dinov2, _DCTFallback
    from distseal.ciphermark.registry import TraceRegistry
    from distseal.ciphermark.wam_ciphermark import (CipherMarkConfig, CipherMarkKeys,
                                                    CipherMarkWam)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    HASH_BITS = 256
    loc = _copie_du_drive(CHEMINS_TATOUEUR[modele])
    if bavard: print(f"      chargement du tatoueur « {modele} »")
    cfg = get_config_from_checkpoint(loc)
    WAM = setup_model_from_checkpoint(loc).to(DEVICE).eval()
    NBITS, IMG_SIZE = int(cfg.args.nbits), int(cfg.args.img_size)
    # Le calendrier de force du filigrane DOIT etre rejoue : sans cela on
    # applique l'amplitude du DEBUT d'entrainement -- 0,5 au lieu de 0,1152 --
    # et l'image sort rayee, avec 8 dB de PSNR.
    if getattr(cfg.args, "scaling_w_schedule", None):
        ck = torch.load(loc, map_location="cpu", weights_only=True)
        if ck.get("epoch") is not None:
            pr = uoptim.parse_params(cfg.args.scaling_w_schedule)
            uoptim.ScalingScheduler(obj=WAM.blender, attribute="scaling_w",
                                    scaling_o=cfg.args.scaling_w, **pr).step(ck["epoch"])
    entraine = float(WAM.blender.scaling_w)
    WAM.blender.scaling_w = float(force)
    if bavard: print(f"      force du filigrane : {entraine:.4f} (entrainee) -> {force:.4f} (appliquee)")
    dino = _try_load_dinov2()
    PHASH = PerceptualHash(n_bits=HASH_BITS, backbone=dino or _DCTFallback()).to(DEVICE).eval()
    CLES, REGISTRE = CipherMarkKeys.random(), TraceRegistry()
    CM = CipherMarkWam(wam=WAM, phash=PHASH, keys=CLES,
                       cfg=CipherMarkConfig(n_bits=NBITS, max_fixed_point_iters=3),
                       registry=REGISTRE)
    TAU = round(CM.cfg.hamming_threshold * HASH_BITS)
    if bavard: print(f"      pret : Omega {NBITS} bits, {IMG_SIZE} px, tau = {TAU}/{HASH_BITS}, {DEVICE}")

def charger_decodeur(nom="phase E — decodeur de SANA (2,4 Go)", bavard=True):
    """Autoencodeur + conditionneur FiLM sur Omega."""
    global AE, COND
    import torch
    if not torch.cuda.is_available():
        raise SystemExit("Le DC-AE exige CUDA (TritonRMSNorm n'a pas d'implementation CPU).\n"
                         "Execution -> Modifier le type d'execution -> GPU.")
    if "NBITS" not in globals(): charger_tatoueur()
    rel, modele = SPEC_DECODEUR[nom]
    loc = _copie_du_drive(rel)
    from deps.efficientvit.ae_model_zoo import DCAE_HF
    from distseal.ciphermark.conditioner import OmegaConditioner, decouvre_etages
    if bavard: print(f"      chargement de {modele}")
    AE = DCAE_HF.from_pretrained(f"mit-han-lab/{modele}").cuda().eval()
    with torch.no_grad():
        etages = decouvre_etages(AE.decoder, AE.encoder(torch.zeros(1,3,256,256).cuda()))
    canaux = [c for _, _, c in etages]
    COND = OmegaConditioner(nbits=NBITS, channels=canaux, gamma_max=0.3, beta_max=0.1).cuda()
    COND.attach([m for _, m, _ in etages])
    ck = torch.load(loc, map_location="cuda", weights_only=False)
    sd = ck.get("state_dict", ck)
    pref = {k.split("omega_conditioner.",1)[1]: v for k, v in sd.items()
            if k.startswith("omega_conditioner.")}
    if not pref:
        raise SystemExit("aucun poids de conditionneur : ce n'est pas un checkpoint conditionne")
    # strict=True : un chargement permissif laisserait une partie du
    # conditionneur a zero, la modulation deviendrait l'identite, et les
    # verdicts seraient faux sans que rien ne le signale.
    COND.load_state_dict(pref, strict=True)
    if bavard:
        print(f"      {len(canaux)} etages {canaux}, "
              f"{COND.n_parametres()/1e6:.2f} M parametres, pas {ck.get('global_step','?')}")

def charger_sana(bavard=True):
    """Pipeline texte vers image. Environ 9 Go au premier passage."""
    global PIPE
    import torch, sys, subprocess, importlib
    if not torch.cuda.is_available(): raise SystemExit("GPU requis pour SANA.")
    manquants = []
    for pq, mod in (("diffusers","diffusers"), ("transformers","transformers"),
                    ("accelerate","accelerate"), ("sentencepiece","sentencepiece")):
        try: __import__(mod)
        except ImportError: manquants.append(pq)
    if manquants:
        if bavard: print(f"      installation de {' '.join(manquants)}")
        subprocess.run([sys.executable,"-m","pip","install","-q",*manquants], check=True)
        importlib.invalidate_caches()
    from diffusers import SanaPipeline
    M = "Efficient-Large-Model/Sana_600M_512px_diffusers"
    if bavard: print(f"      chargement de {M} (9 Go au premier passage, 5 a 10 min)")
    PIPE = SanaPipeline.from_pretrained(M, torch_dtype=DTYPE_AMP)
    libre = (torch.cuda.get_device_properties(0).total_memory
             - torch.cuda.memory_reserved(0)) / 2**30
    if libre < 10:
        if bavard: print(f"      {libre:.1f} Go de VRAM libres -> dechargement vers la RAM")
        PIPE.enable_model_cpu_offload()
    else:
        PIPE = PIPE.to("cuda")
    PIPE.set_progress_bar_config(disable=True)

def assurer(*quoi):
    """Charge a la demande ce qui manque, sans jamais refaire l'existant."""
    if "tatoueur" in quoi and "WAM" not in globals():
        print("  [auto] le tatoueur n'etait pas charge :"); charger_tatoueur()
    if "decodeur" in quoi and "COND" not in globals():
        print("  [auto] le decodeur conditionne n'etait pas charge :"); charger_decodeur()
    if "sana" in quoi and "PIPE" not in globals():
        print("  [auto] SANA n'etait pas charge :"); charger_sana()

print()
print("  chargeurs prets : charger_tatoueur() · charger_decodeur() · charger_sana()")
print("  Les cellules suivantes appellent assurer(...) et chargent ce qui manque")
print("  toutes seules : l'ordre d'execution n'a plus d'importance.")

---
# 2 · Choisir et charger un tatoueur

Quatre couples embedder/extracteur ont été entraînés. Choisissez celui à
essayer : la cellule le télécharge depuis le Drive (une seule fois, il est
ensuite en cache) et l'instancie.

| modèle | ce qu'il est | bit_acc mesurée |
|---|---|---|
| **64 bits** | le modèle de référence, utilisé partout dans le mémoire | 0,9998 |
| **phase B** | le même, réentraîné avec des augmentations | plus robuste : flou k7 de 72,5 % à 98,2 % |
| **96 bits** | canal plus large | défaillant : 85,3 % de détection sans attaque |
| **128 bits** | canal encore plus large | intermédiaire |

In [ ]:
#@title Charger un tatoueur  { display-mode: "form" }
MODELE = "64 bits (reference)" #@param ["64 bits (reference)", "phase B (robuste aux attaques)", "96 bits", "128 bits"]
#@markdown ---
#@markdown **Force du filigrane.** Le checkpoint a fini son entrainement a
#@markdown 0,1152, ce qui donne 22,5 dB de PSNR — un tatouage visible. La
#@markdown descendre a 0,08 gagne 3,2 dB **sans perdre un seul bit**. Sous
#@markdown 0,04 la lecture s'effondre : le modele n'a jamais appris a
#@markdown travailler avec un signal faible, faute de pression de fidelite
#@markdown pendant l'entrainement (`lambda_i = 0`).
#@markdown
#@markdown | valeur | PSNR | bit_acc | verdict |
#@markdown |---|---|---|---|
#@markdown | 0,1152 *(entraine)* | 22,5 dB | 1,000 | rendu |
#@markdown | **0,08** | **25,7 dB** | **1,000** | **rendu** |
#@markdown | 0,06 | 28,2 dB | 0,938 | rendu |
#@markdown | 0,04 | 31,7 dB | 0,813 | limite |
#@markdown | 0,02 | 37,7 dB | 0,531 | illisible |
FORCE_DU_FILIGRANE = 0.08 #@param {type:"slider", min:0.01, max:0.16, step:0.005}

import time
t0 = time.time()
charger_tatoueur(MODELE, FORCE_DU_FILIGRANE)
print(f"\n  pret en {time.time()-t0:.0f} s")
print("  Les cles sont tirees au hasard a chaque execution de cette cellule.")

---
# 3 · Tatouer une image

Chargez une image depuis votre machine, ou laissez la cellule en prendre une
d'exemple. Elle sort marquée, avec son PSNR, son nonce et le hash de référence
enregistré au registre.

**Notez le nonce affiché** : c'est lui qu'il faudra donner à la section 4 pour
vérifier l'image.

In [ ]:
#@title Tatouer  { display-mode: "form" }
SOURCE = "televerser une image" #@param ["televerser une image", "image d'exemple (skimage)"]
IDENTIFIANT_UTILISATEUR = "" #@param {type:"string"}

assurer("tatoueur")
import io as _io, os, numpy as np, torch
from PIL import Image
import matplotlib.pyplot as plt

def charge_image(source):
    if source.startswith("televerser"):
        from google.colab import files
        envoi = files.upload()
        if not envoi: raise SystemExit("aucun fichier recu")
        nom = list(envoi)[0]
        return Image.open(_io.BytesIO(envoi[nom])).convert("RGB"), nom
    from skimage import data
    return Image.fromarray(data.chelsea()).convert("RGB"), "chelsea.png"

img_pil, nom = charge_image(SOURCE)
x = torch.from_numpy(np.asarray(img_pil.resize((IMG_SIZE, IMG_SIZE)))
                     .astype(np.float32) / 255.).permute(2, 0, 1)[None].to(DEVICE)

# Cle derivee d'un identifiant : deux utilisateurs sur la meme image donnent
# des Omega differents, ce qui rend l'attribution opposable.
if IDENTIFIANT_UTILISATEUR.strip():
    from distseal.ciphermark.witness import WitnessField
    CM.witness = WitnessField.for_user(k_master=CLES.k_secret, s_master=CLES.s_master,
                                       user_id=IDENTIFIANT_UTILISATEUR.strip(),
                                       cfg=CM.witness.cfg)
    print(f"cle derivee de l'identifiant « {IDENTIFIANT_UTILISATEUR.strip()} »")

sortie  = CM.embed(x)
marquee = sortie["imgs_w"]
NONCE   = sortie["image_ids"][0]
OMEGA   = sortie["omega"][0].cpu().numpy()

mse  = float(((x - marquee) ** 2).mean())
psnr = 10 * np.log10(1.0 / mse) if mse > 0 else float("inf")
diff = (marquee - x).abs()

fig, ax = plt.subplots(1, 3, figsize=(11, 3.8))
for a, im, t in zip(ax, [x, marquee, diff / max(diff.max().item(), 1e-9)],
                    ["originale", "marquée", "différence (amplifiée)"]):
    a.imshow(im[0].permute(1, 2, 0).cpu().numpy().clip(0, 1)); a.set_title(t, fontsize=9); a.axis("off")
plt.tight_layout(); plt.show()

MARQUEE_GLOBALE = marquee.clone()
os.makedirs("/content/sorties", exist_ok=True)
chemin = f"/content/sorties/marquee_{NONCE}.png"
Image.fromarray((marquee[0].permute(1,2,0).cpu().numpy().clip(0,1)*255).astype(np.uint8)).save(chemin)

print()
print(f"  image            : {nom}")
print(f"  NONCE            : {NONCE}      <-- a reporter en section 4")
repere = ("filigrane VISIBLE" if psnr < 24 else
          "discret" if psnr < 30 else
          "tres discret" if psnr < 36 else "proche de l'etat de l'art")
print(f"  PSNR             : {psnr:.2f} dB   ({repere})")
print(f"  Omega grave      : {''.join(map(str, OMEGA[:32]))}... ({NBITS} bits)")
print(f"  point fixe       : {sortie['n_iters']} iterations, converge={sortie['converged']}")
print(f"  fichier          : {chemin}")
print()
print("  Telechargez-la si vous voulez la reverifier apres l'avoir modifiee")
print("  dans un autre logiciel :")
print("      from google.colab import files ; files.download('%s')" % chemin)

---
# 4 · Extraire et vérifier

Reprenez une image marquée — celle de la section 3, ou une que vous téléversez —
et donnez son nonce. Le vérifieur extrait Ω, recalcule ce qu'il devrait valoir,
compare le hash au contenu, et rend un verdict.

Deux contrôles **indépendants** décident du verdict : l'extraction du témoin, et
la liaison au contenu. La cellule affiche les deux séparément, pour qu'on voie
lequel échoue quand un verdict tombe.

In [ ]:
#@title Extraire et vérifier  { display-mode: "form" }
SOURCE_VERIF = "l'image de la section 3" #@param ["l'image de la section 3", "televerser une image marquee"]
NONCE_A_VERIFIER = -1 #@param {type:"integer"}

assurer("tatoueur")
import io as _io, numpy as np, torch
from PIL import Image

if SOURCE_VERIF.startswith("l'image"):
    y = MARQUEE_GLOBALE
    nonce = NONCE if NONCE_A_VERIFIER < 0 else NONCE_A_VERIFIER
else:
    from google.colab import files
    envoi = files.upload()
    nom = list(envoi)[0]
    im = Image.open(_io.BytesIO(envoi[nom])).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    y = torch.from_numpy(np.asarray(im).astype(np.float32)/255.).permute(2,0,1)[None].to(DEVICE)
    if NONCE_A_VERIFIER < 0:
        raise SystemExit("donnez le NONCE de l'image dans le champ ci-dessus")
    nonce = NONCE_A_VERIFIER

# Les deux controles, separement, avant le verdict global
bits = (WAM.detect(y, **CM._batch_kwargs())["preds"][:, 1:1+NBITS] > 0).to(torch.uint8)[0].cpu().numpy()
h_obs = CM._phash_bytes(y)[0]
try:
    h_ref  = CM.registry.h_ref_for(nonce)
    d_hash = CM._bit_distance(h_obs, h_ref)
except Exception as e:
    h_ref, d_hash = None, None
    print(f"nonce {nonce} absent du registre : {e}")

rapport = CM.verify(y, [nonce])[0]

print()
print("=" * 62)
print(f"  VERDICT : {rapport.verdict.value.upper()}")
print("=" * 62)
print(f"  controle 1 — le temoin Omega")
print(f"      distance   : {rapport.distance} / {rapport.total} bits")
print(f"      p-valeur   : {rapport.p_value:.3e}")
print(f"      confiance  : {rapport.confidence:.4f}")
print(f"  controle 2 — la liaison au contenu")
if d_hash is not None:
    verdict_contenu = "OK" if d_hash <= TAU else "REJETE"
    print(f"      distance du hash : {d_hash} / {HASH_BITS}   (seuil tau = {TAU})")
    print(f"      contenu          : {verdict_contenu}")
else:
    print(f"      non evaluable (nonce inconnu)")
print("=" * 62)
if rapport.verdict.value == "authentic":
    print("  Les deux controles passent : l'image est authentifiee, et")
    print("  l'attribution est opposable au detenteur des cles.")
elif d_hash is not None and d_hash > TAU:
    print("  Le temoin est lisible mais le CONTENU a ete rejete : l'image n'est")
    print("  pas celle qui a ete marquee sous ce nonce. C'est le comportement")
    print("  attendu face a une transplantation de filigrane.")
else:
    print("  Le temoin n'a pas ete retrouve : mauvaise cle, mauvais nonce, ou")
    print("  degradation trop forte de l'image.")

---
# 5 · Attaquer l'image et revérifier

Choisissez une transformation et son intensité. La cellule l'applique à l'image
marquée, puis rejoue la vérification. C'est de quoi retrouver soi-même les
chiffres de robustesse du mémoire — ou les contredire.

Les attaques **actives** sont incluses : la transplantation greffe le filigrane
de votre image sur un contenu étranger, et devrait être refusée.

In [ ]:
#@title Attaquer puis reverifier  { display-mode: "form" }
ATTAQUE = "JPEG" #@param ["aucune", "JPEG", "flou gaussien", "redimensionnement", "recadrage", "bruit gaussien", "luminosite", "contraste", "TRANSPLANTATION (attaque active)"]
INTENSITE = 30 #@param {type:"number"}

assurer("tatoueur")
import numpy as np, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from distseal.augmentation import valuemetric, geometric

AIDE = {"JPEG":"qualite 10-95 (30 = forte compression)",
        "flou gaussien":"taille du noyau, impair : 3, 5, 7",
        "redimensionnement":"facteur 0.1-1.0 (0.5 = moitie)",
        "recadrage":"surface conservee 0.1-1.0 (0.9 = 90 %)",
        "bruit gaussien":"ecart-type 0.01-0.30",
        "luminosite":"facteur (0.8 = plus sombre, 1.2 = plus clair)",
        "contraste":"facteur (0.8 assombri, 1.2 accentue)",
        "TRANSPLANTATION (attaque active)":"sans effet — le residu est greffe sur une autre image"}
print(f"  {ATTAQUE} — INTENSITE : {AIDE.get(ATTAQUE,'')}")

y = MARQUEE_GLOBALE.clone()
if ATTAQUE == "JPEG":
    z = valuemetric.JPEG()(y, None, quality=int(INTENSITE))[0]
elif ATTAQUE == "flou gaussien":
    k = int(INTENSITE) | 1
    z = valuemetric.GaussianBlur()(y, None, kernel_size=k)[0]
elif ATTAQUE == "redimensionnement":
    z = geometric.Resize()(y, None, size=float(INTENSITE))[0]
elif ATTAQUE == "recadrage":
    z = geometric.Crop()(y, None, size=float(INTENSITE))[0]
elif ATTAQUE == "bruit gaussien":
    z = valuemetric.GaussianNoise()(y, None, std=float(INTENSITE))[0]
elif ATTAQUE == "luminosite":
    z = valuemetric.Brightness()(y, None, float(INTENSITE))[0]
elif ATTAQUE == "contraste":
    z = valuemetric.Contrast()(y, None, float(INTENSITE))[0]
elif ATTAQUE.startswith("TRANSPLANTATION"):
    # Le residu du filigrane, greffe sur une AUTRE image, presente sous le
    # nonce d'origine. C'est l'attaque que la liaison au contenu doit bloquer.
    from skimage import data
    from PIL import Image
    autre = Image.fromarray(data.coffee()).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    a = torch.from_numpy(np.asarray(autre).astype(np.float32)/255.).permute(2,0,1)[None].to(DEVICE)
    z = (a + (MARQUEE_GLOBALE - x)).clamp(0, 1)
else:
    z = y
z = z.clamp(0, 1)
if z.shape[-2:] != (IMG_SIZE, IMG_SIZE):
    z = F.interpolate(z, size=(IMG_SIZE, IMG_SIZE), mode="bilinear",
                      align_corners=False, antialias=True)

fig, ax = plt.subplots(1, 2, figsize=(7.5, 3.8))
for a_, im, t in zip(ax, [MARQUEE_GLOBALE, z], ["marquée", f"après {ATTAQUE}"]):
    a_.imshow(im[0].permute(1,2,0).cpu().numpy().clip(0,1)); a_.set_title(t, fontsize=9); a_.axis("off")
plt.tight_layout(); plt.show()

h_obs = CM._phash_bytes(z)[0]
d_hash = CM._bit_distance(h_obs, CM.registry.h_ref_for(NONCE))
r = CM.verify(z, [NONCE])[0]
print()
print(f"  VERDICT            : {r.verdict.value.upper()}")
print(f"  Omega              : {r.distance} / {r.total} bits d'erreur   (p = {r.p_value:.2e})")
print(f"  hash du contenu    : {d_hash} / {HASH_BITS}   seuil tau = {TAU}  ->  "
      f"{'OK' if d_hash <= TAU else 'REJETE'}")
if ATTAQUE.startswith("TRANSPLANTATION"):
    print()
    print("  Attendu : NOT_WATERMARKED. Le temoin reste lisible — le filigrane a")
    print("  bien ete greffe — mais le hash ne correspond plus au contenu.")
    print("  Mesure de reference sur 5000 images : 0,10 % de ces attaques passent.")

---
# 6 · Le décodeur génératif conditionné

**GPU obligatoire.** Ici le tatouage ne s'ajoute pas après coup : le décodeur
du modèle génératif est lui-même modulé par Ω, et produit une image qui porte
son témoin **en sortant du réseau**.

Deux modèles disponibles :

- **phase D** — `dc-ae-f64c128-in-1.0`, autoencodeur ImageNet. 40 000 pas,
  bit_acc médiane 1,0 sur 5000 images générées. Checkpoint de 5,8 Go.
- **phase E** — `dc-ae-f32c32-sana-1.0`, le décodeur latent de SANA. 40 000 pas,
  maximum 0,9867. Checkpoint de 2,4 Go. C'est celui qu'utilise la section 8.

La cellule encode votre image en latent, la décode **sous un Ω de votre choix**,
puis relit le témoin dans l'image produite.

In [ ]:
#@title Charger un décodeur conditionné et l'essayer  { display-mode: "form" }
DECODEUR = "phase E — decodeur de SANA (2,4 Go)" #@param ["phase E — decodeur de SANA (2,4 Go)", "phase D — autoencodeur ImageNet (5,8 Go)"]

import time
t0 = time.time()
assurer("tatoueur")
charger_decodeur(DECODEUR)
print(f"\n  pret en {time.time()-t0:.0f} s")

In [ ]:
#@title Générer une image porteuse d'Ω, puis relire le témoin  { display-mode: "form" }
OMEGA_CHOISI = "aleatoire" #@param ["aleatoire", "tout a zero", "tout a un", "alterne 0101"]
SOURCE_IMAGE = "reutiliser celle de la section 3" #@param ["reutiliser celle de la section 3", "image d'exemple (skimage)", "televerser une image"]

assurer("tatoueur", "decodeur")
import io as _io, torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image

# Cette cellule doit pouvoir tourner SANS avoir execute la section 3 : on
# recupere donc son image si elle existe, sinon on en charge une ici.
def _image_de_travail():
    if SOURCE_IMAGE.startswith("reutiliser") and "x" in globals():
        return globals()["x"].to(DEVICE), "celle de la section 3"
    if SOURCE_IMAGE.startswith("televerser"):
        from google.colab import files
        envoi = files.upload()
        if not envoi: raise SystemExit("aucun fichier recu")
        nom = list(envoi)[0]
        im = Image.open(_io.BytesIO(envoi[nom])).convert("RGB")
        src = nom
    else:
        from skimage import data
        im = Image.fromarray(data.chelsea()).convert("RGB")
        src = "chelsea.png (skimage)"
        if SOURCE_IMAGE.startswith("reutiliser"):
            src += "  [la section 3 n'a pas ete executee]"
    a = np.asarray(im.resize((IMG_SIZE, IMG_SIZE))).astype(np.float32) / 255.
    return torch.from_numpy(a).permute(2, 0, 1)[None].to(DEVICE), src

x, _origine = _image_de_travail()
print(f"  image : {_origine}")

if OMEGA_CHOISI == "aleatoire":
    om = torch.randint(0, 2, (1, NBITS), device="cuda")
elif OMEGA_CHOISI == "tout a zero":
    om = torch.zeros(1, NBITS, dtype=torch.long, device="cuda")
elif OMEGA_CHOISI == "tout a un":
    om = torch.ones(1, NBITS, dtype=torch.long, device="cuda")
else:
    om = torch.tensor([[i % 2 for i in range(NBITS)]], device="cuda")

xin = (x.cuda() * 2 - 1)   # le modele attend [-1, 1]
with torch.no_grad(), torch.autocast("cuda", dtype=DTYPE_AMP):
    with COND.omega(om):
        y_cond, _, _ = AE(xin, global_step=0)
    y_nu, _, _ = AE(xin, global_step=0)          # sans conditionnement
y_cond = (y_cond.float()*0.5+0.5).clamp(0,1)
y_nu   = (y_nu.float()*0.5+0.5).clamp(0,1)

bits = (WAM.detect(y_cond, **CM._batch_kwargs())["preds"][:,1:1+NBITS] > 0).to(torch.uint8)[0].cpu().numpy()
vrai = om[0].cpu().numpy().astype(np.uint8)
err  = int((bits != vrai).sum())
from distseal.ciphermark.equation import binomial_pvalue
pv = binomial_pvalue(err, NBITS)

fig, ax = plt.subplots(1, 3, figsize=(11, 3.8))
d = (y_cond - y_nu).abs(); d = d / max(d.max().item(), 1e-9)
for a_, im, t in zip(ax, [y_nu, y_cond, d],
                     ["décodée SANS Ω", "décodée AVEC Ω", "ce que Ω a changé"]):
    a_.imshow(im[0].permute(1,2,0).cpu().numpy().clip(0,1)); a_.set_title(t, fontsize=9); a_.axis("off")
plt.tight_layout(); plt.show()

print()
print(f"  Omega demande  : {''.join(map(str, vrai[:40]))}...")
print(f"  Omega relu     : {''.join(map(str, bits[:40]))}...")
print(f"  erreurs        : {err} / {NBITS} bits   ->  bit_acc {1-err/NBITS:.4f}")
print(f"  p-valeur       : {pv:.3e}")
print()
print("  Le temoin n'a JAMAIS ete applique a l'image : il a module les etages du")
print("  decodeur, et l'extracteur le retrouve dans le resultat. C'est la")
print("  difference entre un tatouage ajoute et un tatouage NE dans la generation.")

---
# 7 · L'injection dans l'espace latent

**GPU obligatoire.** Toutes les sections précédentes tatouent des **pixels**.
Ici le filigrane est écrit dans le **latent**, avant décodage.

C'est le seul volet du projet dont le résultat est franchement négatif, et la
cellule le montre plutôt que de le cacher. Le facteur limitant est géométrique :
à 256 px le latent fait 32 × 8 × 8 = 2048 valeurs pour 64 bits, soit **1 pixel
par bit** contre 1024 en pixels. La phase A avait établi que ce ratio gouverne
le plafond.

Maximum atteint après 242 époques : **0,6296**, très loin des 0,99 nécessaires
pour qu'un verdict soit fiable.

Une précision qui compte : l'embedder standard **ne peut pas** tatouer un
latent. Il traite la luminance et n'a qu'un canal d'entrée ; il refuse un
tenseur à 32 canaux. Il a fallu un embedder distinct, `unet_small2_latent32`.

In [ ]:
#@title Essayer l'injection latente (phase G)  { display-mode: "form" }
# Cette section ne charge PAS de checkpoint, et c'est deliberé.
#
# Le checkpoint de la phase G a l'epoque 242 a ete perdu : la sauvegarde
# periodique l'a copie vers le Drive PENDANT que le trainer l'ecrivait, ce qui
# a produit une archive zip tronquee -- torch.load la refuse avec
# "failed finding central directory". Il ne subsiste qu'un instantane a
# l'epoque 12, sans interet.
#
# La perte est reelle mais limitee : ce qui compte scientifiquement, c'est la
# COURBE, et le journal des 243 epoques est intact. La cellule la rapporte, et
# demontre en direct le blocage qui a rendu cette phase necessaire.
import os, torch, yaml
from omegaconf import OmegaConf
from distseal.models.embedder import build_embedder

print("  LE BLOCAGE, demontre en trois lignes")
print("  " + "-"*58)
conf = yaml.safe_load(open("configs/embedder.yaml"))
pixel  = build_embedder("unet_small2_yuv_quant",
                        OmegaConf.create(conf["unet_small2_yuv_quant"]), NBITS)
latent = build_embedder("unet_small2_latent32",
                        OmegaConf.create(conf["unet_small2_latent32"]), NBITS)
z = torch.randn(2, 32, 8, 8); msg = torch.randint(0, 2, (2, NBITS))
print("  embedder PIXEL sur un latent a 32 canaux :")
try:
    pixel(z, msg); print("      accepte (inattendu)")
except RuntimeError as e:
    print(f"      REFUSE — {str(e)[:78]}")
print("  embedder LATENT sur le meme tenseur :")
o = latent(z, msg)
print(f"      accepte, sortie {tuple(o.shape)}, "
      f"{sum(p.numel() for p in latent.parameters())/1e6:.2f} M parametres")

print()
print("  CE QUE LA PHASE G A MESURE — 243 epoques sur 1200")
print("  " + "-"*58)
COURBE_G = [(0,0.5017,32.45),(54,0.5780,26.18),(108,0.5725,26.11),
            (162,0.6147,25.97),(189,0.6272,25.95),(242,0.6225,26.03)]
print(f"  {'epoque':>8}{'bit_acc':>10}{'psnr':>9}")
for e,b,q in COURBE_G: print(f"  {e:>8}{b:>10.4f}{q:>9.2f}")
print(f"  --> maximum 0,6296 a l'epoque 191   (seuil necessaire : 0,99)")

print()
print("  POURQUOI ELLE PLAFONNE — c'est geometrique")
print("  " + "-"*58)
print(f"  {'espace':<26}{'latent':>16}{'valeurs':>10}{'val/bit':>10}")
for nom, lat, v in (("pixel (phase A)","256x256 x 1",65536),
                    ("MaskGIT (DistSeal)","16x16 x 256",65536),
                    ("DC-AE f32c32 (phase G)","8x8 x 32",2048)):
    print(f"  {nom:<26}{lat:>16}{v:>10}{v//64:>10}")
print()
print("  DistSeal a choisi un autoencodeur dont le latent conserve autant de")
print("  valeurs que l'image en pixels. La phase G en avait TRENTE-DEUX FOIS")
print("  moins. La phase A avait etabli que ce ratio gouverne le plafond.")
print("  La phase I rejoue donc l'experience sur MaskGIT, a 1024 valeurs/bit.")

---
# 8 · Du prompt à l'image tatouée

**GPU obligatoire, et environ 12 Go de téléchargement la première fois.**

C'est la chaîne complète : vous écrivez une phrase, SANA la rend en image, et
Ω est gravé **pendant le décodage** — pas après. Puis le vérifieur rend son
verdict.

Deux modes, et ils ne disent pas la même chose :

- **`posthoc`** — SANA génère, *ensuite* CipherMark tatoue. Ça marche, mais le
  tatouage reste une étape ajoutée, qu'un opérateur du modèle peut sauter.
- **`inmodel`** — le décodeur latent est conditionné (phase E). Le tatouage
  naît dans la génération, il n'y a rien à sauter.

Modifiez le prompt autant que vous voulez et relancez la cellule.

In [ ]:
#@title  ①  Charger SANA (facultatif : la cellule du prompt le fait aussi)  { display-mode: "form" }
import time
t0 = time.time()
charger_sana()
print(f"\n  SANA pret en {time.time()-t0:.0f} s")

In [ ]:
#@title  ②  Écrivez votre prompt, puis exécutez  (SANA doit etre charge)  { display-mode: "form" }
PROMPT = "un phare dans la tempete, photographie" #@param {type:"string"}
MODE = "inmodel — le tatouage nait dans le decodage" #@param ["inmodel — le tatouage nait dans le decodage", "posthoc — SANA genere puis on tatoue"]
ETAPES_DE_DIFFUSION = 14 #@param {type:"slider", min:4, max:40, step:2}
GUIDAGE = 4.5 #@param {type:"number"}
GRAINE = 0 #@param {type:"integer"}
ATTAQUE_APRES = "aucune" #@param ["aucune", "JPEG q50", "JPEG q30"]

import torch, numpy as np, torch.nn.functional as F, matplotlib.pyplot as plt, time
inmodel = MODE.startswith("inmodel")
assurer("tatoueur", "sana", *(("decodeur",) if inmodel else ()))

g = torch.Generator(device="cuda").manual_seed(int(GRAINE))
t0 = time.time()
with torch.no_grad():
    img = PIPE(prompt=PROMPT, num_inference_steps=int(ETAPES_DE_DIFFUSION),
               guidance_scale=float(GUIDAGE), generator=g,
               height=512, width=512, output_type="pt").images
t_gen = time.time() - t0
img = img.float().clamp(0, 1).cuda()
xw = F.interpolate(img, size=(IMG_SIZE, IMG_SIZE), mode="bilinear",
                   align_corners=False, antialias=True)

if inmodel:
    om = torch.randint(0, 2, (1, NBITS), device="cuda")
    with torch.no_grad(), torch.autocast("cuda", dtype=DTYPE_AMP):
        with COND.omega(om):
            yc, _, _ = AE((xw * 2 - 1), global_step=0)
    finale = (yc.float()*0.5+0.5).clamp(0,1)
    bits = (WAM.detect(finale, **CM._batch_kwargs())["preds"][:,1:1+NBITS] > 0).to(torch.uint8)[0].cpu().numpy()
    err = int((bits != om[0].cpu().numpy().astype(np.uint8)).sum())
    from distseal.ciphermark.equation import binomial_pvalue
    pv, d_hash, verdict = binomial_pvalue(err, NBITS), None, None
else:
    sortie = CM.embed(xw)
    finale = sortie["imgs_w"]; nonce = sortie["image_ids"][0]
    r = CM.verify(finale, [nonce])[0]
    err, pv, verdict = r.distance, r.p_value, r.verdict.value
    d_hash = CM._bit_distance(CM._phash_bytes(finale)[0], CM.registry.h_ref_for(nonce))

if ATTAQUE_APRES != "aucune":
    from distseal.augmentation import valuemetric
    q = 50 if "q50" in ATTAQUE_APRES else 30
    attaquee = valuemetric.JPEG()(finale.clone(), None, quality=q)[0].clamp(0,1)
else:
    attaquee = None

n = 3 if attaquee is not None else 2
fig, ax = plt.subplots(1, n, figsize=(4*n, 4))
vues = [(img, "SANA brut, 512 px"), (finale, "porteuse de Ω")]
if attaquee is not None: vues.append((attaquee, ATTAQUE_APRES))
for a_, (im, t) in zip(np.atleast_1d(ax), vues):
    a_.imshow(im[0].permute(1,2,0).cpu().numpy().clip(0,1)); a_.set_title(t, fontsize=9); a_.axis("off")
plt.suptitle(f'« {PROMPT} »', fontsize=10); plt.tight_layout(); plt.show()

print(f"  genere en {t_gen:.1f} s   mode : {'IN-MODEL' if inmodel else 'POST-HOC'}")
print(f"  Omega        : {err} / {NBITS} bits d'erreur   ->  bit_acc {1-err/NBITS:.4f}")
print(f"  p-valeur     : {pv:.3e}")
if verdict:  print(f"  VERDICT      : {verdict.upper()}")
if d_hash is not None:
    print(f"  hash contenu : {d_hash} / {HASH_BITS}  (seuil {TAU})  -> {'OK' if d_hash<=TAU else 'REJETE'}")
if attaquee is not None:
    b2 = (WAM.detect(attaquee, **CM._batch_kwargs())["preds"][:,1:1+NBITS] > 0).to(torch.uint8)[0].cpu().numpy()
    ref = om[0].cpu().numpy().astype(np.uint8) if inmodel else None
    if ref is not None:
        print(f"  apres {ATTAQUE_APRES} : {int((b2!=ref).sum())} / {NBITS} bits d'erreur")

---
# Ce que ce banc d'essai permet de vérifier soi-même

- qu'un tatouage **survit** à la compression, au flou, au redimensionnement, et
  qu'il **meurt** au recadrage au-delà de 10 % de surface perdue ;
- qu'une **transplantation** de filigrane est refusée — mais pas toujours : la
  mesure de référence sur 5000 images donne 0,10 % de réussites ;
- qu'un décodeur génératif peut porter un témoin **différent à chaque
  génération**, ce qu'aucune méthode par distillation ne permet ;
- que l'**injection latente** apprend sans atteindre l'exploitable, pour une
  raison géométrique identifiée ;
- qu'un **prompt** suffit à produire une image dont la provenance est
  cryptographiquement vérifiable.

Les chiffres de référence, mesurés sur 5000 images, sont dans le carnet
`CipherMark-presentation-des-resultats.ipynb`.